# Phase 40 — Improved TF-IDF + SVM

Βελτίωση του TF-IDF + SVM με:
1. GridSearch για C και ngram_range
2. Character n-grams
3. Feature union (word + char n-grams)
4. Voting ensemble πολλών SVM

**Baseline:** TF-IDF + SVM (tuned) = 0.7419 (Kaggle)

In [2]:
import numpy as np
import pandas as pd
import re
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score
from sklearn.pipeline import Pipeline, FeatureUnion
from itertools import product
import warnings
warnings.filterwarnings('ignore')

In [3]:
train = pd.read_csv('train.csv')
valid = pd.read_csv('valid.csv')
test  = pd.read_csv('test.csv')

def preprocess(text):
    if not isinstance(text, str): return ''
    text = text.lower()
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

train['combined'] = train['title'].apply(preprocess) + ' ' + train['text'].str[:550].apply(preprocess)
valid['combined'] = valid['title'].apply(preprocess) + ' ' + valid['text'].str[:550].apply(preprocess)
test['combined']  = test['title'].apply(preprocess)  + ' ' + test['text'].str[:550].apply(preprocess)

print(f'Train: {len(train)} | Valid: {len(valid)} | Test: {len(test)}')


def official_st1_score(y_true_h, y_pred_h, y_true_p, y_pred_p, verbose=True):
    f1_h = f1_score(y_true_h, y_pred_h, average='macro', zero_division=0)
    mask = (np.array(y_true_h) == np.array(y_pred_h))
    f1_p = f1_score(
        np.array(y_true_p)[mask], np.array(y_pred_p)[mask],
        average='macro', zero_division=0
    ) if mask.sum() > 0 else 0.0
    score = (f1_h + f1_p) / 2
    if verbose:
        print(f'  macro-F1 Hazard:                    {f1_h:.4f}')
        print(f'  Σωστά hazard:                       {mask.sum()}/{len(mask)} ({100*mask.mean():.1f}%)')
        print(f'  macro-F1 Product (given correct h): {f1_p:.4f}')
        print(f'  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
        print(f'  OFFICIAL ST1 SCORE:                 {score:.4f}')
    return score


def train_eval_svm(X_train, X_valid, C, label_train, label_valid, verbose=False):
    clf = LinearSVC(C=C, class_weight='balanced', max_iter=2000, random_state=42)
    clf.fit(X_train, label_train)
    preds = clf.predict(X_valid)
    f1 = f1_score(label_valid, preds, average='macro', zero_division=0)
    return clf, preds, f1

Train: 5082 | Valid: 565 | Test: 997


## 1. GridSearch — C και ngram_range

In [4]:
C_values     = [0.1, 0.5, 1.0, 5.0, 10.0]
ngram_values = [(1,1), (1,2), (1,3)]
max_features = 50000

print('=== GRIDSEARCH: C x ngram_range ===\n')
print(f'{"ngram":10s} {"C":6s} {"F1 Hazard":12s} {"F1 Product":12s} {"ST1 Score":10s}')
print('-' * 55)

best_score  = 0
best_params = {}
results     = []

for ngram in ngram_values:
    tfidf = TfidfVectorizer(
        max_features=max_features,
        ngram_range=ngram,
        sublinear_tf=True,
        stop_words='english'
    )
    X_train = tfidf.fit_transform(train['combined'])
    X_valid = tfidf.transform(valid['combined'])

    for C in C_values:
        clf_h, pred_h, f1_h = train_eval_svm(X_train, X_valid, C,
                                               train['hazard-category'],
                                               valid['hazard-category'])
        clf_p, pred_p, f1_p = train_eval_svm(X_train, X_valid, C,
                                               train['product-category'],
                                               valid['product-category'])
        score = official_st1_score(
            valid['hazard-category'], pred_h,
            valid['product-category'], pred_p,
            verbose=False
        )
        results.append({'ngram': ngram, 'C': C, 'f1_h': f1_h, 'f1_p': f1_p, 'score': score})
        print(f'{str(ngram):10s} {C:<6.1f} {f1_h:<12.4f} {f1_p:<12.4f} {score:.4f}')

        if score > best_score:
            best_score  = score
            best_params = {'ngram': ngram, 'C': C, 'tfidf': tfidf}

print(f'\nΒέλτιστο: ngram={best_params["ngram"]}, C={best_params["C"]} → ST1={best_score:.4f}')
print(f'Baseline: 0.7419')

=== GRIDSEARCH: C x ngram_range ===

ngram      C      F1 Hazard    F1 Product   ST1 Score 
-------------------------------------------------------
(1, 1)     0.1    0.7696       0.6175       0.6806
(1, 1)     0.5    0.7977       0.7360       0.7589
(1, 1)     1.0    0.8042       0.7466       0.7676
(1, 1)     5.0    0.8060       0.7303       0.7623
(1, 1)     10.0   0.8047       0.7284       0.7606
(1, 2)     0.1    0.7602       0.5674       0.6463
(1, 2)     0.5    0.8040       0.6979       0.7436
(1, 2)     1.0    0.8132       0.6825       0.7415
(1, 2)     5.0    0.7951       0.7371       0.7615
(1, 2)     10.0   0.7917       0.7407       0.7609
(1, 3)     0.1    0.7622       0.5036       0.6151
(1, 3)     0.5    0.8054       0.6641       0.7250
(1, 3)     1.0    0.8147       0.6748       0.7352
(1, 3)     5.0    0.7956       0.6883       0.7354
(1, 3)     10.0   0.7935       0.6778       0.7278

Βέλτιστο: ngram=(1, 1), C=1.0 → ST1=0.7676
Baseline: 0.7419


## 2. Character N-grams

In [5]:
print('=== CHARACTER N-GRAMS ===\n')

char_ngrams = [(2,4), (3,5), (2,5)]

for char_ngram in char_ngrams:
    tfidf_char = TfidfVectorizer(
        max_features=30000,
        ngram_range=char_ngram,
        analyzer='char_wb',  # char_wb: respects word boundaries
        sublinear_tf=True
    )
    X_train_char = tfidf_char.fit_transform(train['combined'])
    X_valid_char = tfidf_char.transform(valid['combined'])

    clf_h, pred_h, _ = train_eval_svm(X_train_char, X_valid_char, 0.5,
                                       train['hazard-category'], valid['hazard-category'])
    clf_p, pred_p, _ = train_eval_svm(X_train_char, X_valid_char, 0.5,
                                       train['product-category'], valid['product-category'])
    score = official_st1_score(
        valid['hazard-category'], pred_h,
        valid['product-category'], pred_p,
        verbose=False
    )
    print(f'char_wb {char_ngram}: {score:.4f}')

print(f'\nBaseline (word ngrams): 0.7419')

=== CHARACTER N-GRAMS ===

char_wb (2, 4): 0.7446
char_wb (3, 5): 0.7328
char_wb (2, 5): 0.7339

Baseline (word ngrams): 0.7419


## 3. Feature Union — Word + Char N-grams

In [6]:
print('=== FEATURE UNION: Word + Char N-grams ===\n')

# Βέλτιστο word ngram από GridSearch
best_ngram = best_params['ngram']
best_C     = best_params['C']

tfidf_word = TfidfVectorizer(
    max_features=50000,
    ngram_range=best_ngram,
    sublinear_tf=True,
    stop_words='english'
)
tfidf_char = TfidfVectorizer(
    max_features=30000,
    ngram_range=(3, 5),
    analyzer='char_wb',
    sublinear_tf=True
)

X_train_word = tfidf_word.fit_transform(train['combined'])
X_valid_word = tfidf_word.transform(valid['combined'])
X_test_word  = tfidf_word.transform(test['combined'])

X_train_char = tfidf_char.fit_transform(train['combined'])
X_valid_char = tfidf_char.transform(valid['combined'])
X_test_char  = tfidf_char.transform(test['combined'])

# Concatenation
X_train_union = sp.hstack([X_train_word, X_train_char])
X_valid_union = sp.hstack([X_valid_word, X_valid_char])
X_test_union  = sp.hstack([X_test_word,  X_test_char])

print(f'Word features:  {X_train_word.shape[1]:,}')
print(f'Char features:  {X_train_char.shape[1]:,}')
print(f'Union features: {X_train_union.shape[1]:,}\n')

# Δοκιμή διαφορετικών C
best_union_score = 0
best_union_C     = 0.5

for C in C_values:
    clf_h, pred_h, _ = train_eval_svm(X_train_union, X_valid_union, C,
                                       train['hazard-category'], valid['hazard-category'])
    clf_p, pred_p, _ = train_eval_svm(X_train_union, X_valid_union, C,
                                       train['product-category'], valid['product-category'])
    score = official_st1_score(
        valid['hazard-category'], pred_h,
        valid['product-category'], pred_p,
        verbose=False
    )
    print(f'C={C}: {score:.4f}')
    if score > best_union_score:
        best_union_score = score
        best_union_C     = C

print(f'\nΒέλτιστο Union: C={best_union_C} → {best_union_score:.4f}')

=== FEATURE UNION: Word + Char N-grams ===

Word features:  12,695
Char features:  30,000
Union features: 42,695

C=0.1: 0.7049
C=0.5: 0.7697
C=1.0: 0.7718
C=5.0: 0.7648
C=10.0: 0.7592

Βέλτιστο Union: C=1.0 → 0.7718


## 4. Voting Ensemble πολλών SVM

In [7]:
from sklearn.preprocessing import MinMaxScaler

print('=== VOTING ENSEMBLE ΠΟΛΛΩΝ SVM ===\n')

# Configurations: (tfidf, C)
configs = [
    ('word (1,2) C=0.5', X_train_word, X_valid_word, X_test_word,  0.5),
    ('word (1,2) C=1.0', X_train_word, X_valid_word, X_test_word,  1.0),
    ('union C=best',     X_train_union, X_valid_union, X_test_union, best_union_C),
]

# Συλλογή decision scores από κάθε SVM
all_scores_h_valid = []
all_scores_p_valid = []
all_scores_h_test  = []
all_scores_p_test  = []
svm_classes_h = None
svm_classes_p = None

for name, X_tr, X_va, X_te, C in configs:
    clf_h = LinearSVC(C=C, class_weight='balanced', max_iter=2000, random_state=42)
    clf_h.fit(X_tr, train['hazard-category'])
    clf_p = LinearSVC(C=C, class_weight='balanced', max_iter=2000, random_state=42)
    clf_p.fit(X_tr, train['product-category'])

    # Normalize decision scores [0,1]
    sc = MinMaxScaler()
    all_scores_h_valid.append(sc.fit_transform(clf_h.decision_function(X_va)))
    all_scores_h_test.append(sc.transform(clf_h.decision_function(X_te)))

    sc2 = MinMaxScaler()
    all_scores_p_valid.append(sc2.fit_transform(clf_p.decision_function(X_va)))
    all_scores_p_test.append(sc2.transform(clf_p.decision_function(X_te)))

    svm_classes_h = clf_h.classes_
    svm_classes_p = clf_p.classes_
    print(f'Trained: {name}')

# Average των scores
avg_scores_h_valid = np.mean(all_scores_h_valid, axis=0)
avg_scores_p_valid = np.mean(all_scores_p_valid, axis=0)
avg_scores_h_test  = np.mean(all_scores_h_test,  axis=0)
avg_scores_p_test  = np.mean(all_scores_p_test,  axis=0)

ensemble_pred_h = svm_classes_h[avg_scores_h_valid.argmax(axis=1)]
ensemble_pred_p = svm_classes_p[avg_scores_p_valid.argmax(axis=1)]

print('\n=== ΑΠΟΤΕΛΕΣΜΑΤΑ VOTING ENSEMBLE ===')
score_ensemble = official_st1_score(
    valid['hazard-category'], ensemble_pred_h,
    valid['product-category'], ensemble_pred_p
)

=== VOTING ENSEMBLE ΠΟΛΛΩΝ SVM ===

Trained: word (1,2) C=0.5
Trained: word (1,2) C=1.0
Trained: union C=best

=== ΑΠΟΤΕΛΕΣΜΑΤΑ VOTING ENSEMBLE ===
  macro-F1 Hazard:                    0.6928
  Σωστά hazard:                       514/565 (91.0%)
  macro-F1 Product (given correct h): 0.5391
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  OFFICIAL ST1 SCORE:                 0.6160


## 5. Σύνοψη & Submission

In [8]:
print('\n=== ΣΥΝΟΨΗ ===')
print(f'Baseline TF-IDF+SVM:          0.7419')
print(f'GridSearch βέλτιστο:           {best_score:.4f}  (ngram={best_params["ngram"]}, C={best_params["C"]})')
print(f'Feature Union (word+char):     {best_union_score:.4f}')
print(f'Voting Ensemble:               {score_ensemble:.4f}')

# Submission με το καλύτερο
all_scores = {
    'gridsearch': best_score,
    'union':      best_union_score,
    'ensemble':   score_ensemble
}
best_method = max(all_scores, key=all_scores.get)
print(f'\nΒέλτιστη μέθοδος: {best_method} ({all_scores[best_method]:.4f})')

# Submission voting ensemble
test_pred_h = svm_classes_h[avg_scores_h_test.argmax(axis=1)]
test_pred_p = svm_classes_p[avg_scores_p_test.argmax(axis=1)]

pd.DataFrame({
    'id': test['id'],
    'hazard-category': test_pred_h,
    'product-category': test_pred_p
}).to_csv('submission_svm_improved.csv', index=False)
print('\nΑποθηκεύτηκε: submission_svm_improved.csv')


=== ΣΥΝΟΨΗ ===
Baseline TF-IDF+SVM:          0.7419
GridSearch βέλτιστο:           0.7676  (ngram=(1, 1), C=1.0)
Feature Union (word+char):     0.7718
Voting Ensemble:               0.6160

Βέλτιστη μέθοδος: union (0.7718)

Αποθηκεύτηκε: submission_svm_improved.csv


In [9]:
# Submission με Feature Union (word + char) — βέλτιστο C
print(f'=== SUBMISSION Feature Union (word+char), C={best_union_C} ===\n')

clf_h_union = LinearSVC(C=best_union_C, class_weight='balanced', max_iter=2000, random_state=42)
clf_h_union.fit(X_train_union, train['hazard-category'])

clf_p_union = LinearSVC(C=best_union_C, class_weight='balanced', max_iter=2000, random_state=42)
clf_p_union.fit(X_train_union, train['product-category'])

pd.DataFrame({
    'id': test['id'],
    'hazard-category':  clf_h_union.predict(X_test_union),
    'product-category': clf_p_union.predict(X_test_union)
}).to_csv('submission_feature_union.csv', index=False)

print(f'Αποθηκεύτηκε: submission_feature_union.csv')
print(f'Validation score: {best_union_score:.4f}')

=== SUBMISSION Feature Union (word+char), C=1.0 ===

Αποθηκεύτηκε: submission_feature_union.csv
Validation score: 0.7718
